In [ ]:
#imports
%load_ext autoreload
%autoreload 2

import numpy as np
import src.viz_utils as viz_utils
import src.utils as utils
import importlib
from src.utils import CanonPart, CanonPartMetadata, get_pointcloud_in_cam_frame, transform_cloud_to_base, remove_outliers
from PIL import Image
import torch
import open3d as o3d
import open3d.visualization.gui as gui
import copy as cp
import pickle
import matplotlib

from src.object_warping import (
    ObjectWarpingSE2Batch,
    ObjectSE2Batch,
    ObjectSE3Batch,
    ObjectWarpingSE3Batch,
    warp_to_pcd,
    warp_to_pcd_se2,
    warp_to_pcd_se3,
    warp_to_pcd_se3_hemisphere,
    PARAM_1,
    ALIGNMENT_PARAM,
    mask_and_cost_batch_pt,
)

In [ ]:
#load pcl 
save_name = '/home/rthomp12/fewshot/red_mug_thin_rack_test/whole_init_scene_pcls.npz'
masked_pcls = np.load(save_name)

In [ ]:
#set objects and parts 

# Mug v Rack
parent_object = 'rack'
child_object = 'mug'

parent_model_files = {'rack': }
child_model_files = {'mug':}


# # Teapot v Mug

# parent_object = 'mug'
# child_object = 'teapot'

# parent_model_files = {'mug': }
# child_model_files = {'teapot':}


# # Bowl v Mug

# parent_object = 'mug'
# child_object = 'bowl'

# parent_model_files = {'mug': }
# child_model_files = {'teapot': }


parent_part_models = {part: CanonPart.from_pickle(parent_part_model_files[part]) for part in parent_part_names}
child_part_models = {part: CanonPart.from_pickle(child_part_model_files[part]) for part in child_part_names}


In [ ]:
# Warping parameters 
n_angles = 8

PARAM_1 = {"lr": 1e-2, 
           "n_steps": 200,
           "n_samples": 1000, 
           "object_size_reg": 0.1} #.01

inference_kwargs = {
                            "train_latents": True,
                            "train_scales": True,
                            "train_poses": True,
                        }

In [ ]:
#warping

device = 'cuda'

canon = parent_part_models[parent_object]

# enables optimization incorporating the relational descriptors
cost_function = (
    lambda source, target, canon_part_labels: mask_and_cost_batch_pt(
        target,
        parent_part_labels[parent_object],
        source,
        canon_part_labels,
    )
)

warp = ObjectWarpingSE3Batch(
        canon,
        masked_pcls[parent_object],
        device,
        canon_labels=canon_parent_part_labels[target_part],
        cost_function=cost_function,
        **cp.deepcopy(PARAM_1),
    )
parent_reconstruction, _, parent_params = warp_to_pcd_se3(
    warp, n_angles, n_batches=12, inference_kwargs=inference_kwargs
)


print(target_part)
canon = child_part_models[child_object]

# enables optimization incorporating the relational descriptors
cost_function = (
    lambda source, target, canon_part_labels: mask_and_cost_batch_pt(
        target,
        child_part_labels[target_part],
        source,
        canon_part_labels,
    )
)

warp = ObjectWarpingSE3Batch(
        canon,
        masked_pcls[child_object],
        device,
        canon_labels=canon_child_part_labels[target_part],
        cost_function=cost_function,
        **cp.deepcopy(PARAM_1),
    )
child_reconstruction, _, child_params = warp_to_pcd_se3(
    warp, n_angles, n_batches=12, inference_kwargs=inference_kwargs
)

    

In [ ]:
all_reconstructions = {child_object: child_reconstruction, parent_object: parent_reconstruction}    
viz_utils.show_pcds_plotly({f'reconstructed_{part}': all_reconstructions[part] for part in all_reconstructions.keys()} | 
                           {key: masked_pcls[key] for key in masked_pcls.keys() if len(masked_pcls[key]) > 0})

In [ ]:
save_name = "/home/rthomp12/fewshot/blue_mug_thin_rack_demo/whole_initial_scene_warps"
np.savez(save_name, 
         child_reconstructions=child_reconstructions,
         parent_reconstructions=parent_reconstructions,
         child_params=child_params, 
         parent_params=parent_params,
         )

In [ ]:
# Warping and saving meshes
import trimesh

child_mesh = child_part_model.to_transformed_mesh(child_params) 
parent_mesh = parent_part_models.to_transformed_mesh(parent_params) 
child_mesh.export('/home/rthomp12/fewshot/blue_mug_thin_rack/child.obj')
parent_mesh.export('/home/rthomp12/fewshot/blue_mug_thin_rack/parent.obj')

utils.convex_decomposition(child_mesh, '/home/rthomp12/fewshot/blue_mug_thin_rack/child_cd.obj')
utils.convex_decomposition(child_mesh, '/home/rthomp12/fewshot/blue_mug_thin_rack/parent_cd.obj')